<a href="https://colab.research.google.com/github/charmy-patel/practicals/blob/bigdata/PIG_HIVE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Step 1–3 → This is like Pig: data cleansing, transformation, filtering.

Step 4–7 → This is like Hive: structured queries, aggregations, saving as Hive tables for BI tools.

In [27]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, sum as spark_sum, avg

# 1️⃣ Start Spark with Hive support
spark = SparkSession.builder \
    .appName("PigHiveExample") \
    .enableHiveSupport() \
    .getOrCreate()

# 2️⃣ Load raw data (CSV for example)
raw_df = spark.read.csv("/content/drive/My Drive/Colab Notebooks/clickstream.csv", header=True, inferSchema=True)

# Sample columns: user_id, timestamp, page, device_type, purchase_amount

# 3️⃣ Data Cleaning & Transformation (Pig-like)
clean_df = raw_df \
    .filter(col("purchase_amount").isNotNull()) \
    .withColumn("timestamp", to_timestamp(col("timestamp"), "dd-MM-yyyy HH:mm")) \
    .filter(col("timestamp").isNotNull()) \
    .filter(col("device_type").isin("mobile", "desktop"))  # only keep valid devices


# 4️⃣ Save cleaned data into Hive table
clean_df.write.mode("overwrite").saveAsTable("ecommerce_cleaned")

# 5️⃣ Run HiveQL (Hive-like)

result_df = spark.sql("""
    SELECT *
    FROM ecommerce_cleaned
   """)

# 6️⃣ Show results
result_df.show()

# 7️⃣ Save results back to Hive
result_df.write.mode("overwrite").saveAsTable("ecommerce_sales_summary")

!ls -l "/content/drive/My Drive/Colab Notebooks/clickstream.csv"
!head -n "/content/drive/My Drive/Colab Notebooks/clickstream.csv"

+-------+-------------------+--------+-----------+---------------+
|user_id|          timestamp|    page|device_type|purchase_amount|
+-------+-------------------+--------+-----------+---------------+
|user_14|2025-01-22 04:15:00|    home|    desktop|         146.04|
| user_7|2025-01-09 10:36:00|checkout|     mobile|          78.21|
|user_39|2025-02-27 19:07:00| product|    desktop|          41.05|
|user_37|2025-03-07 08:18:00|    cart|    desktop|         134.53|
|user_41|2025-01-15 13:29:00|checkout|     mobile|          83.11|
|user_34|2025-01-23 21:13:00|    home|     mobile|         139.52|
| user_9|2025-03-02 01:14:00|checkout|    desktop|          60.36|
|user_18|2025-03-02 21:24:00|checkout|    desktop|         112.45|
|user_16|2025-02-13 01:13:00|checkout|    desktop|          27.92|
|user_44|2025-03-01 11:29:00|    home|     mobile|           86.5|
| user_6|2025-03-10 14:11:00| product|    desktop|          87.22|
|user_14|2025-02-06 12:05:00|    home|    desktop|           8

In [29]:
result_df = spark.sql("""
    SELECT device_type,
          SUM(purchase_amount) AS TotalRevenue
    FROM ecommerce_cleaned
    GROUP BY device_type
   """).show()

+-----------+------------------+
|device_type|      TotalRevenue|
+-----------+------------------+
|    desktop|1697.8999999999996|
|     mobile|2315.7699999999995|
+-----------+------------------+



In [32]:
result_df = spark.sql("""
    SELECT device_type,
           COUNT(DISTINCT user_id) AS unique_users,
           SUM(purchase_amount) AS total_sales,
           AVG(purchase_amount) AS avg_purchase
    FROM ecommerce_cleaned
    GROUP BY device_type
   """).show()

+-----------+------------+------------------+------------------+
|device_type|unique_users|       total_sales|      avg_purchase|
+-----------+------------+------------------+------------------+
|    desktop|          17|1697.8999999999996| 80.85238095238094|
|     mobile|          16|           2315.77|121.88263157894737|
+-----------+------------+------------------+------------------+

